# House Price Prediction – Linear Regression

**Objective:** Build a regression pipeline with one-hot encoding, evaluate predictions and interpret coefficients.

> Dataset note: included CSVs are synthetic practice data created for this portfolio. Replace with approved public datasets for official internship submission.


## Setup
Run the following cell. Install missing packages with `pip install -r ../../requirements.txt` from the repository root.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression,Ridge
from sklearn.metrics import mean_squared_error,r2_score
df=pd.read_csv('house_prices_synthetic.csv'); display(df.head()); display(df.describe())
df.Price.plot(kind='hist', bins=30, title='House price distribution', figsize=(8,4)); plt.xlabel('Price'); plt.show()
X=df.drop(columns='Price'); y=df.Price
num=X.select_dtypes(include='number').columns; cat=X.select_dtypes(exclude='number').columns
pre=ColumnTransformer([('num',SimpleImputer(strategy='median'),num),
 ('cat',Pipeline([('imputer',SimpleImputer(strategy='most_frequent')),('onehot',OneHotEncoder(handle_unknown='ignore'))]),cat)])
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.2,random_state=42)
for name,reg in [('Linear Regression',LinearRegression()),('Ridge',Ridge(alpha=1.0))]:
    pipe=Pipeline([('prep',pre),('model',reg)]); pipe.fit(Xtr,ytr); pred=pipe.predict(Xte)
    print(name,'MSE',mean_squared_error(yte,pred),'RMSE',mean_squared_error(yte,pred)**.5,'R2',r2_score(yte,pred))
    if name=='Linear Regression':
        plt.scatter(yte,pred,alpha=.6); lim=[min(yte.min(),pred.min()),max(yte.max(),pred.max())]
        plt.plot(lim,lim,'r--'); plt.xlabel('Actual'); plt.ylabel('Predicted'); plt.title('Actual vs predicted'); plt.show()
        residual=yte-pred; plt.scatter(pred,residual); plt.axhline(0,color='red',ls='--'); plt.title('Residual plot'); plt.show()
        names=pipe.named_steps['prep'].get_feature_names_out()
        display(pd.Series(pipe.named_steps['model'].coef_,index=names).sort_values())
print('Interpret coefficients in original units for numeric predictors; one-hot coefficients are relative to the dropped/reference category.')
